# Lesson 12a: Fine-tuning and Adaptation — Theory

9a/9b built attention and 10a/10b assembled it into the Transformer — a generic
architecture trained on a generic objective (predict masked or next tokens) over a
large, task-agnostic corpus. Nothing about that pretraining objective mentions
*sentiment*, or *translation*, or whatever concrete problem a practitioner actually
has. This lesson asks the question that follows immediately from having a
pretrained network at all: given a model that already knows a great deal in
general, how do you specialise it to a small labelled task cheaply, and what do
you risk losing when you do?

## Introduction

Three families of answer are in common use, and they trade cost, data
requirement and final accuracy off against each other:

- **Prompting** — use the pretrained model exactly as it is. No parameter is
  touched; the model's existing knowledge is queried directly on the new input.
  Cheapest option: zero labelled examples, zero training compute.
- **Feature extraction** — freeze the pretrained backbone entirely and train
  only a small new head on top of its (fixed) representations. A handful of
  labelled examples is enough, because only the head's parameters need to move.
- **Full fine-tuning** — unfreeze everything and train the whole network,
  backbone included, on the downstream task. Most flexible and most accurate
  given enough data, most expensive, and — the risk this lesson spends its
  second half on — most likely to overwrite whatever the backbone previously
  knew.

The running example throughout is a small sentiment classifier. A backbone (a
two-layer network mapping a bag-of-words vector to a hidden representation) is
pretrained once on a labelled sentiment task, "Task A". A second sentiment
task, "Task B", drawn over the *same vocabulary* but with an independently
chosen set of sentiment-bearing words, stands in for a shifted domain a
practitioner adapts to later — the kind of shift between, say, film reviews and
product reviews, where some words that carry sentiment in one barely appear as
signal in the other. Every method below is compared on how well it adapts the
Task-A-pretrained backbone to Task B, and later, on how much of Task A it
costs to do so.

## Setup

In [ ]:
# Fixed seeds: data generation, weight initialisation, and gradient descent are
# all deterministic given these.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

## Adaptation Strategies

A pretrained backbone here is a small MLP: a hidden layer
$h = \tanh(W_0 x + b_0)$ mapping a $V$-dimensional bag-of-words count vector to
an $H$-dimensional representation, followed by a linear head
$\hat{y} = \sigma(w_h^\top h + b_h)$ producing a sentiment probability.
$\theta = \{W_0, b_0, w_h, b_h\}$ denotes all of it.

The vocabulary is $V$ synthetic word-ids. Task A designates a random $n_\text{pos}$
of them as positive indicators and a disjoint $n_\text{neg}$ as negative
indicators; a synthetic "review" is generated by drawing $L$ words from a
distribution that favours positive-indicator words when the true label is
positive and negative-indicator words when it is negative. Task B repeats this
with an *independently drawn* assignment of which words are the indicators —
same kind of task, different words doing the signalling.

In [ ]:
N_POS, N_NEG, N_NEUTRAL = 20, 20, 20
VOCAB_SIZE = N_POS + N_NEG + N_NEUTRAL   # V
HIDDEN = 32                              # H
DOC_LEN = 12                             # words per synthetic "review"
TEMP = 0.4                               # sharpness of the label-conditional word distribution


def true_weights(rng, V, n_pos, n_neg, scale=1.0):
    """A random n_pos words get weight +scale (positive indicators), a disjoint
    n_neg get -scale (negative indicators), the rest are neutral (weight 0)."""
    idx = rng.permutation(V)
    w = np.zeros(V)
    w[idx[:n_pos]] = scale
    w[idx[n_pos:n_pos + n_neg]] = -scale
    return w


def word_probs(w, temp):
    """Label-conditional word distribution: a word with weight +w is
    exp(2w/temp) times more likely in a positive review than a negative one."""
    p_pos = np.exp(w / temp)
    p_neg = np.exp(-w / temp)
    return p_pos / p_pos.sum(), p_neg / p_neg.sum()


def generate_task(rng, w_true, n, flip_prob=0.05):
    V = w_true.shape[0]
    prob_pos, prob_neg = word_probs(w_true, TEMP)
    y = rng.integers(0, 2, size=n).astype(np.float64)
    x = np.zeros((n, V))
    for i in range(n):
        probs = prob_pos if y[i] == 1 else prob_neg
        word_idx = rng.choice(V, size=DOC_LEN, p=probs)
        x[i] = np.bincount(word_idx, minlength=V)
    flips = rng.uniform(size=n) < flip_prob   # label noise
    y[flips] = 1.0 - y[flips]
    return x, y


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


def forward(x, W0, b0, w_head, b_head):
    hidden = np.tanh(x @ W0.T + b0)
    return hidden, sigmoid(hidden @ w_head + b_head)


def accuracy(prob, y):
    return float(np.mean((prob > 0.5) == (y > 0.5)))


def backward(x, y, hidden, prob, w_head):
    n = x.shape[0]
    dlogit = (prob - y) / n                    # dL/dlogit, (n,)
    dw_head = hidden.T @ dlogit                 # (H,)
    db_head = dlogit.sum()
    dhidden = np.outer(dlogit, w_head)          # (n, H)
    dpre = dhidden * (1 - hidden ** 2)          # tanh'
    dW0 = dpre.T @ x                            # (H, V)
    db0 = dpre.sum(axis=0)                      # (H,)
    return dW0, db0, dw_head, db_head


def train_full(x, y, W0, b0, w_head, b_head, iters=400, lr=0.08):
    W0, b0, w_head = W0.copy(), b0.copy(), w_head.copy()
    for _ in range(iters):
        hidden, prob = forward(x, W0, b0, w_head, b_head)
        dW0, db0, dw_head, db_head = backward(x, y, hidden, prob, w_head)
        W0 -= lr * dW0
        b0 -= lr * db0
        w_head -= lr * dw_head
        b_head -= lr * db_head
    return W0, b0, w_head, b_head


rng = np.random.default_rng(SEED)
w_true_A = true_weights(rng, VOCAB_SIZE, N_POS, N_NEG)
w_true_B = true_weights(rng, VOCAB_SIZE, N_POS, N_NEG)   # independent word roles

x_A_train, y_A_train = generate_task(rng, w_true_A, n=200)
x_A_test, y_A_test = generate_task(rng, w_true_A, n=200)
x_B_test, y_B_test = generate_task(rng, w_true_B, n=200)

W0_init = rng.normal(scale=0.2, size=(HIDDEN, VOCAB_SIZE))
b0_init = np.zeros(HIDDEN)
w_head_init = rng.normal(scale=0.2, size=HIDDEN)

W0_A, b0_A, w_head_A, b_head_A = train_full(
    x_A_train, y_A_train, W0_init, b0_init, w_head_init, 0.0
)
_, prob_A_test = forward(x_A_test, W0_A, b0_A, w_head_A, b_head_A)
acc_A_baseline = accuracy(prob_A_test, y_A_test)
print(f"pretrained backbone -- Task A test accuracy: {acc_A_baseline:.3f}")

With a pretrained backbone in hand, adapt it to Task B under each of the
three strategies. Prompting reuses $\theta$ untouched; feature extraction
freezes $(W_0, b_0)$ and trains a fresh head; full fine-tuning starts from
$\theta$ and updates everything. Sweeping the number of labelled Task B
examples available (averaging 6 random training draws at each size, to smooth
out small-sample noise) shows the strategies' very different data appetites.

In [ ]:
def train_head_only(x, y, W0, b0, w_head, b_head, iters=400, lr=0.5):
    """Feature extraction: (W0, b0) held fixed, only the head moves."""
    w_head = w_head.copy()
    for _ in range(iters):
        hidden, prob = forward(x, W0, b0, w_head, b_head)
        dlogit = (prob - y) / x.shape[0]
        w_head -= lr * (hidden.T @ dlogit)
        b_head -= lr * dlogit.sum()
    return w_head, b_head


# Prompting / zero-shot: apply the Task-A-pretrained model to Task B untouched.
_, prob_prompt = forward(x_B_test, W0_A, b0_A, w_head_A, b_head_A)
acc_prompt = accuracy(prob_prompt, y_B_test)

REPEATS = 6
n_train_grid = [5, 10, 20, 40, 80]
acc_feature_extraction, acc_full_finetune = [], []
acc_A_after_full_repeats = []   # kept for the Catastrophic Forgetting section (n_train=40 only)

for n_train in n_train_grid:
    fe_accs, ft_accs = [], []
    for rep in range(REPEATS):
        rng_n = np.random.default_rng(1000 * n_train + rep)
        x_B_train, y_B_train = generate_task(rng_n, w_true_B, n=n_train)

        w_head_fe, b_head_fe = train_head_only(
            x_B_train, y_B_train, W0_A, b0_A, rng_n.normal(scale=0.2, size=HIDDEN), 0.0
        )
        _, prob_fe = forward(x_B_test, W0_A, b0_A, w_head_fe, b_head_fe)
        fe_accs.append(accuracy(prob_fe, y_B_test))

        W0_ft, b0_ft, w_head_ft, b_head_ft = train_full(
            x_B_train, y_B_train, W0_A, b0_A, w_head_A, b_head_A
        )
        _, prob_ft = forward(x_B_test, W0_ft, b0_ft, w_head_ft, b_head_ft)
        ft_accs.append(accuracy(prob_ft, y_B_test))

        if n_train == 40:
            _, prob_A_full = forward(x_A_test, W0_ft, b0_ft, w_head_ft, b_head_ft)
            acc_A_after_full_repeats.append(accuracy(prob_A_full, y_A_test))

    acc_feature_extraction.append(float(np.mean(fe_accs)))
    acc_full_finetune.append(float(np.mean(ft_accs)))

print(f"prompting (zero-shot) Task B accuracy: {acc_prompt:.3f}")
for n, fe, ft in zip(n_train_grid, acc_feature_extraction, acc_full_finetune):
    print(f"  n_train={n:>3}  feature-extraction={fe:.3f}  full-fine-tune={ft:.3f}")

plt.figure()
plt.axhline(acc_prompt, color="gray", linestyle="--", label="prompting (zero-shot)")
plt.plot(n_train_grid, acc_feature_extraction, "o-", label="feature extraction")
plt.plot(n_train_grid, acc_full_finetune, "o-", label="full fine-tuning")
plt.xlabel("labelled Task B examples")
plt.ylabel("Task B test accuracy (mean of 6 draws)")
plt.title("Adaptation strategy vs. labelled data available")
plt.legend()
plt.tight_layout()
plt.show()

Prompting is not merely weak here, it is actively misleading: the frozen
head's decision boundary happens to correlate *negatively* with Task B's
labels, so zero-shot transfer scores well below chance. A stale model applied
to a sufficiently different domain does not degrade gracefully to
"uninformative" — it can be confidently wrong, which is its own argument
against trusting a frozen model on a domain nobody checked it against. Feature
extraction reaches strong accuracy from a handful of examples and barely
improves with more, because it is only fitting a single linear head. Full
fine-tuning starts behind feature extraction when data is very scarce (more
parameters need more evidence to move correctly) but catches up as more
labelled data becomes available — at the cost of touching every backbone
weight, which is exactly what makes it expensive and, as the *Catastrophic
Forgetting* section below shows, risky.

**When to use which:** prompting when no labelled data exists at all or
immediate deployment matters more than accuracy — and only after checking it
is not actively wrong, which it can be; feature extraction when labelled data
is scarce, compute is limited, or the backbone must keep serving other tasks
unchanged; full fine-tuning when labelled data is plentiful and maximum
accuracy on this one task outweighs the cost and the risk of forgetting.

## LoRA Derived

Full fine-tuning of a weight matrix $W \in \mathbb{R}^{d_{\text{out}} \times
d_{\text{in}}}$ learns an unconstrained update $\Delta W$, costing
$d_{\text{out}} d_{\text{in}}$ trainable parameters — for the backbone above,
$H \times V$. **LoRA** (Hu et al., 2021) hypothesises that the update a
downstream task actually needs has much lower intrinsic rank than the matrix
itself, and enforces that directly by factorising it:

$$W = W_0 + \Delta W, \qquad \Delta W = BA, \qquad
B \in \mathbb{R}^{d_{\text{out}} \times r},\ A \in \mathbb{R}^{r \times d_{\text{in}}},\
r \ll \min(d_{\text{out}}, d_{\text{in}})$$

$W_0$ (the pretrained weight) stays frozen; only $B$ and $A$ train, at a cost
of $r(d_{\text{out}} + d_{\text{in}})$ parameters — the *Parameter Efficiency*
section below counts exactly how much smaller that is. $B$ is initialised to
all zeros and $A$ to small random values, so $\Delta W = 0$ at the start of
training and adaptation begins from the pretrained model's exact behaviour.

For a linear layer $y = (W_0 + BA)x$ under a loss $L$, the chain rule gives the
gradient of $L$ with respect to the *would-be* full update,
$\partial L/\partial W = (\partial L/\partial y)\,x^\top$, exactly as in full
fine-tuning — and one further application of the chain rule through
$\Delta W = BA$ gives the LoRA gradients:

$$\frac{\partial L}{\partial B} = \frac{\partial L}{\partial W} A^\top,
\qquad
\frac{\partial L}{\partial A} = B^\top \frac{\partial L}{\partial W}$$

The same chain rule extends unchanged through whatever nonlinearity and head
sit downstream of $W_0$ — the implementation below applies it to the sentiment
backbone's hidden layer exactly as it applied to $W_0$ itself for full
fine-tuning.

In [ ]:
# Verify the LoRA gradient formulas above against PyTorch autograd on a small,
# self-contained linear layer before trusting them inside the full backbone.
rng_chk = np.random.default_rng(SEED)
d_out, d_in, r_chk = 6, 8, 2
W0_chk = rng_chk.normal(size=(d_out, d_in))
B_chk = rng_chk.normal(scale=0.1, size=(d_out, r_chk))
A_chk = rng_chk.normal(scale=0.1, size=(r_chk, d_in))
x_chk = rng_chk.normal(size=d_in)
target_chk = rng_chk.normal(size=d_out)

# Manual: y = (W0 + BA) x,  L = 1/2 ||y - target||^2
y_chk = (W0_chk + B_chk @ A_chk) @ x_chk
dL_dy = y_chk - target_chk
dL_dW = np.outer(dL_dy, x_chk)          # same formula as full fine-tuning
dL_dB_manual = dL_dW @ A_chk.T
dL_dA_manual = B_chk.T @ dL_dW

# Autograd on the identical computation.
W0_t = torch.tensor(W0_chk)
B_t = torch.tensor(B_chk, requires_grad=True)
A_t = torch.tensor(A_chk, requires_grad=True)
x_t = torch.tensor(x_chk)
target_t = torch.tensor(target_chk)
y_t = (W0_t + B_t @ A_t) @ x_t
loss_t = 0.5 * ((y_t - target_t) ** 2).sum()
loss_t.backward()

print("max |dL/dB manual - autograd|:", np.abs(dL_dB_manual - B_t.grad.numpy()).max())
print("max |dL/dA manual - autograd|:", np.abs(dL_dA_manual - A_t.grad.numpy()).max())

The derived formulas match PyTorch's autograd to floating-point precision,
so the same update rule can be trusted to train LoRA adapters on the sentiment
backbone.

In [ ]:
def lora_forward(x, W0, b0, B, A, w_head, b_head):
    W_eff = W0 + B @ A
    hidden = np.tanh(x @ W_eff.T + b0)
    return hidden, sigmoid(hidden @ w_head + b_head)


def train_lora(x, y, W0, b0, B, A, w_head, b_head, iters=400, lr=0.15):
    B, A, w_head = B.copy(), A.copy(), w_head.copy()
    for _ in range(iters):
        hidden, prob = lora_forward(x, W0, b0, B, A, w_head, b_head)
        n = x.shape[0]
        dlogit = (prob - y) / n
        dhidden = np.outer(dlogit, w_head)
        dpre = dhidden * (1 - hidden ** 2)
        dW_eff = dpre.T @ x                 # (H, V), same formula as full fine-tuning
        dB = dW_eff @ A.T
        dA = B.T @ dW_eff
        B -= lr * dB
        A -= lr * dA
        w_head -= lr * (hidden.T @ dlogit)
        b_head -= lr * dlogit.sum()
    return B, A, w_head, b_head


def make_lora_params(rng, rank):
    B = np.zeros((HIDDEN, rank))            # zero init: Delta W = 0 at start
    A = rng.normal(scale=0.1, size=(rank, VOCAB_SIZE))
    return B, A


rank_default = 4
lora_accs_default, acc_A_after_lora_repeats = [], []
for rep in range(REPEATS):
    rng_n = np.random.default_rng(1000 * 40 + rep)
    x_B_train, y_B_train = generate_task(rng_n, w_true_B, n=40)
    B0, A0 = make_lora_params(rng_n, rank_default)
    w_head_lora0 = rng_n.normal(scale=0.2, size=HIDDEN)
    B_l, A_l, w_head_l, b_head_l = train_lora(
        x_B_train, y_B_train, W0_A, b0_A, B0, A0, w_head_lora0, 0.0
    )
    _, prob_l = lora_forward(x_B_test, W0_A, b0_A, B_l, A_l, w_head_l, b_head_l)
    lora_accs_default.append(accuracy(prob_l, y_B_test))
    _, prob_A_lora = lora_forward(x_A_test, W0_A, b0_A, B_l, A_l, w_head_l, b_head_l)
    acc_A_after_lora_repeats.append(accuracy(prob_A_lora, y_A_test))

acc_lora_default = float(np.mean(lora_accs_default))
full_acc_40 = acc_full_finetune[n_train_grid.index(40)]
print(f"LoRA (rank={rank_default}) Task B test accuracy: {acc_lora_default:.3f}")
print(f"  vs. full fine-tuning at the same n_train=40:  {full_acc_40:.3f}")

## Parameter Efficiency

Every rank $r$ trades trainable parameters for representational
flexibility. For the backbone's $H \times V$ weight matrix, full fine-tuning
always costs $H \cdot V$ parameters regardless of task difficulty; LoRA costs
$r(H+V)$ — linear in $r$ rather than quadratic in the matrix's dimensions, and
cheaper than full fine-tuning whenever $r < \frac{HV}{H+V}$. Sweep $r$ on the
same Task B adaptation from the previous section to see both sides of that
trade-off: how the parameter count grows, and how much of full fine-tuning's
accuracy a given rank actually recovers.

In [ ]:
full_backbone_params = HIDDEN * VOCAB_SIZE
crossover_rank = full_backbone_params / (HIDDEN + VOCAB_SIZE)

ranks = [1, 2, 4, 8, 16, 32]
lora_params, lora_accs = [], []
for r in ranks:
    accs = []
    for rep in range(REPEATS):
        rng_n = np.random.default_rng(1000 * 40 + rep)
        x_B_train, y_B_train = generate_task(rng_n, w_true_B, n=40)
        B_r, A_r = make_lora_params(rng_n, r)
        w_head_r0 = rng_n.normal(scale=0.2, size=HIDDEN)
        B_r, A_r, w_head_r, b_head_r = train_lora(
            x_B_train, y_B_train, W0_A, b0_A, B_r, A_r, w_head_r0, 0.0
        )
        _, prob_r = lora_forward(x_B_test, W0_A, b0_A, B_r, A_r, w_head_r, b_head_r)
        accs.append(accuracy(prob_r, y_B_test))
    lora_params.append(r * (HIDDEN + VOCAB_SIZE))
    lora_accs.append(float(np.mean(accs)))

print(f"crossover rank (LoRA = full fine-tune params): {crossover_rank:.1f}")
for r, p, a in zip(ranks, lora_params, lora_accs):
    print(f"rank={r:>2}  trainable backbone params={p:>4} "
          f"({100 * p / full_backbone_params:5.1f}% of full)  Task B accuracy={a:.3f}")
print(f"full fine-tuning: trainable backbone params={full_backbone_params}  "
      f"Task B accuracy={full_acc_40:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(ranks, lora_params, "o-", label="LoRA")
ax1.axhline(full_backbone_params, color="gray", linestyle="--", label="full fine-tuning")
ax1.set_xlabel("LoRA rank r")
ax1.set_ylabel("trainable backbone parameters")
ax1.set_title("Parameter count vs. rank")
ax1.legend()

ax2.plot(ranks, lora_accs, "o-", label="LoRA")
ax2.axhline(full_acc_40, color="gray", linestyle="--", label="full fine-tuning")
ax2.set_xlabel("LoRA rank r")
ax2.set_ylabel("Task B test accuracy (mean of 6 draws)")
ax2.set_title("Accuracy vs. rank")
ax2.legend()
plt.tight_layout()
plt.show()

Even the smallest rank tested here already matches full fine-tuning's
accuracy — the low-rank hypothesis LoRA is built on holds comfortably for this
task, at a small fraction of the parameter count. Past the crossover rank
printed above, the factorisation costs *more* parameters than just
fine-tuning the whole matrix, so larger ranks stop being a win at all; picking
$r$ is a real trade-off, not "bigger is always safer".

## Catastrophic Forgetting

Every method above was scored only on Task B. But full fine-tuning did not
train on any Task A data after the initial pretraining — nothing in its
gradient descent objective protects Task A performance, so there is no
guarantee the updated backbone still classifies Task A correctly. This is
**catastrophic forgetting**: adapting a network to a new task can silently
destroy its performance on a previous one, because ordinary gradient descent
has no term that penalises moving away from the old solution.

Re-evaluate the pretrained model, the fully fine-tuned model, and the
LoRA-adapted model — all already trained above on Task B at $n_\text{train}=40$
(or not at all, in the pretrained case) — on the original **Task A** test set,
to measure exactly how much each strategy forgot.

In [ ]:
acc_A_after_full = float(np.mean(acc_A_after_full_repeats))
acc_A_after_lora = float(np.mean(acc_A_after_lora_repeats))
# Feature extraction never touched (W0_A, b0_A) or the original head -- Task A
# is scored with the exact same (backbone, head) pair as the pretrained baseline.
acc_A_after_feature_extraction = acc_A_baseline

methods = ["pretrained\n(no adaptation)", "full fine-tune",
           f"LoRA (r={rank_default})", "feature extraction"]
accs_A = [acc_A_baseline, acc_A_after_full, acc_A_after_lora, acc_A_after_feature_extraction]

for m, a in zip(methods, accs_A):
    print(f"{m.replace(chr(10), ' '):<30} Task A accuracy after Task B adaptation: {a:.3f}")

plt.figure()
plt.bar(methods, accs_A, color=["gray", "tab:red", "tab:orange", "tab:blue"])
plt.axhline(acc_A_baseline, color="black", linestyle=":", linewidth=1)
plt.ylabel("Task A test accuracy (mean of 6 draws where adapted)")
plt.title("Catastrophic forgetting after adapting to Task B")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

Full fine-tuning's Task A accuracy drops measurably: every backbone weight
was free to move toward Task B, including the ones Task A's decision boundary
depended on. LoRA's drop is **larger**, not smaller — despite training barely
a fifth of the parameters full fine-tuning does. Forced to explain Task B's
independently-assigned word roles through only a rank-$r$ subspace of the same
shared hidden representation, LoRA makes large, concentrated changes in
exactly the few directions available to it, and those directions overlap
heavily with the ones Task A's decision boundary also depends on. Constraining
an update's *rank* shrinks its parameter count, but says nothing about *which*
directions it is allowed to touch — a small update aimed squarely at a shared
bottleneck can disturb a previous task more than a large, diffuse one spread
thinly across many redundant weights. Feature extraction is the one strategy
here with a **provable** guarantee, not just an empirically smaller number:
its backbone weights are bit-for-bit identical to the pretrained model, Task A
is scored with its own original head, and nothing about the Task A computation
changed at all.

**The general mitigation** is not "use a low-rank update" — this experiment
shows that alone is not enough — but "freeze what a previous task depends
on". Feature extraction achieves that exactly, by leaving the shared backbone
untouched and adding a task-specific head. A rank-constrained update only
helps when the subspace it is confined to happens not to overlap with the
previous task's important directions, which is a property that has to be
checked, not assumed.

## Key Takeaways

- **Three adaptation strategies trade cost for accuracy**: prompting (free,
  and capable of scoring *below* chance if the domain has shifted enough to
  make the frozen decision boundary actively wrong), feature extraction
  (cheap, needs little labelled data, freezes the backbone), full fine-tuning
  (most accurate given enough data, most expensive, touches every weight).
- **LoRA factorises the update**: $\Delta W = BA$ with $B$ zero-initialised,
  trains $r(d_{\text{out}}+d_{\text{in}})$ parameters instead of
  $d_{\text{out}} d_{\text{in}}$, and its gradient is the same
  full-fine-tuning gradient $\partial L/\partial W$ passed one more
  chain-rule step through $B$ and $A$ — verified here against PyTorch autograd
  to floating-point precision.
- **Parameter savings can be dramatic**: for this backbone, even the smallest
  rank tested already matched full fine-tuning's accuracy; past a computable
  crossover rank, though, the factorisation costs more parameters than just
  fine-tuning the matrix outright.
- **Catastrophic forgetting is a real, measured cost of adaptation** — and a
  smaller parameter count does not automatically mean less of it: LoRA
  forgot *more* of Task A here than full fine-tuning did, because its update
  was concentrated in exactly the directions Task A also depended on.
- **The only mitigation with a proof, not just a hope, is freezing what the
  old task depends on**: feature extraction leaves the backbone bit-for-bit
  unchanged, so its Task A accuracy cannot move at all; a rank-constrained
  update helps only when its subspace happens to avoid the old task's
  important directions.